#### P4-05
# Decision Tree Classification Model
#### M2 - Amri B.S.M
#### IT24102955

## 1. Objective

Develop and validate a Decision Tree classifier for hotel booking cancellation using the canonical shared-prepared data created by `07_shared_prep.ipynb`.

- Use only the **prepared training split** for baseline assessment, 5-fold stratified cross-validation and hyperparameter tuning.
- One-hot encode categorical variables inside the model pipeline; do not scale numerical variables.
- Tune `max_depth` and `min_samples_leaf` using `scoring="roc_auc"` on the agreed training folds.
- Use `GridSearchCV(refit=True)` to fit the selected pipeline on **all existing training rows**.
- Evaluate that fitted model **once on the separate prepared validation split** with `src.evaluate.evaluate()` and an unchanged 0.5 threshold for descriptive label metrics.
- Save the trained model with `joblib`, export readable decision rules, and record CV and validation results separately.

**The test dataset is never loaded or used in this notebook.** Final model comparison and threshold selection are reserved for later phases.

## 2. Colab setup and GitHub repository

This setup is safe to rerun. If `/content/hotel-cancellation` exists without a `.git` directory, the folder is renamed to a backup instead of being deleted. A valid, clean Git checkout is updated with a fast-forward pull; local tracked edits are protected.

In [1]:
from pathlib import Path
import subprocess
import sys
import importlib

REPO_URL = "https://github.com/MahdhiS/hotel-cancellation.git"
ROOT = Path("/content/hotel-cancellation")

# Preserve any non-Git folder rather than deleting its contents.
if ROOT.exists() and not (ROOT / ".git").is_dir():
    backup = Path("/content/hotel-cancellation_backup")
    counter = 1
    while backup.exists():
        backup = Path(f"/content/hotel-cancellation_backup_{counter}")
        counter += 1
    ROOT.rename(backup)
    print("Non-Git folder preserved at:", backup)

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
else:
    changes = subprocess.run(
        ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"],
        check=True, capture_output=True, text=True
    ).stdout.strip()
    if changes:
        print("Local tracked changes detected. Git pull skipped to preserve them.")
    else:
        update = subprocess.run(
            ["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"],
            capture_output=True, text=True
        )
        if update.returncode != 0:
            raise RuntimeError(
                "Could not fast-forward the local repository. Your files were not deleted. "
                "Inspect git status and the error below before continuing:\n" + update.stderr
            )
        print(update.stdout.strip())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()

DATA_DIR = ROOT / "data" / "processed" / "shared_prepared"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for filename in ("hotel_train_prepared.csv", "hotel_val_prepared.csv"):
    filepath = DATA_DIR / filename
    if not filepath.is_file():
        raise FileNotFoundError(f"Missing prepared file: {filepath}. Check GitHub/P3-05 outputs.")

print("Repository:", ROOT)
print("Prepared dataset directory:", DATA_DIR)
print("Output model directory:", MODEL_DIR)

Repository: /content/hotel-cancellation
Prepared dataset directory: /content/hotel-cancellation/data/processed/shared_prepared
Output model directory: /content/hotel-cancellation/models


In [2]:
import numpy as np
import pandas as pd
import joblib
from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV

from src.preprocessing import split_classifier_data
from src.evaluate import evaluate

pd.set_option("display.max_columns", None)
print("Project and model imports successful.")

Project and model imports successful.


## 3. Load the prepared training and validation splits

Feature engineering, leakage removal, missing-value handling, and rare-category grouping were completed in Notebook 07. Load its finalized outputs without running those transformations again. The validation data is held aside during CV and tuning.

In [3]:
train_prepared = pd.read_csv(DATA_DIR / "hotel_train_prepared.csv")
val_prepared = pd.read_csv(DATA_DIR / "hotel_val_prepared.csv")

print("Train shape:", train_prepared.shape)
print("Validation shape:", val_prepared.shape)
print("Train missing values:", int(train_prepared.isna().sum().sum()))
print("Validation missing values:", int(val_prepared.isna().sum().sum()))

assert list(train_prepared.columns) == list(val_prepared.columns)
assert not train_prepared.isna().any().any()
assert not val_prepared.isna().any().any()
train_prepared.head()

Train shape: (58192, 29)
Validation shape: (10458, 29)
Train missing values: 0
Validation missing values: 0


,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,company,adr,total_nights,total_guests,has_kids,season,arrival_day_of_week
0,2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,Direct,Direct,0,0,0,A,No Deposit,0.0,0.0,75.0,1,1.0,0,Summer,Wednesday
1,3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,Corporate,Corporate,0,0,0,A,No Deposit,Other,0.0,75.0,1,1.0,0,Summer,Wednesday
2,4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,240.0,0.0,98.0,2,2.0,0,Summer,Wednesday
3,6,Resort Hotel,0,0,2015,July,27,1,0,2,2,0.0,0,BB,Direct,Direct,0,0,0,C,No Deposit,0.0,0.0,107.0,2,2.0,0,Summer,Wednesday
4,7,Resort Hotel,0,9,2015,July,27,1,0,2,2,0.0,0,FB,Direct,Direct,0,0,0,C,No Deposit,Other,0.0,103.0,2,2.0,0,Summer,Wednesday


In [4]:
X_train, y_train, train_booking_ids = split_classifier_data(train_prepared)
X_val, y_val, val_booking_ids = split_classifier_data(val_prepared)

assert list(X_train.columns) == list(X_val.columns)
assert train_booking_ids.is_unique and val_booking_ids.is_unique
assert set(train_booking_ids).isdisjoint(set(val_booking_ids))
assert set(y_train.unique()) == {0, 1}
assert set(y_val.unique()) == {0, 1}

for forbidden in ("booking_id", "is_canceled", "country"):
    assert forbidden not in X_train.columns
    assert forbidden not in X_val.columns

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)
print("Train cancellation proportion:", round(float(y_train.mean()), 4))
print("Validation cancellation proportion:", round(float(y_val.mean()), 4))
print("Train/validation schema, ID, and leakage checks passed.")

X_train: (58192, 27) | y_train: (58192,)
X_val: (10458, 27) | y_val: (10458,)
Train cancellation proportion: 0.2533
Validation cancellation proportion: 0.2964
Train/validation schema, ID, and leakage checks passed.


In [5]:
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 16
Categorical features: 11


## 4. Decision Tree pipeline and shared CV setup

One-hot encoding for categorical variables is fitted **inside each training fold**. Numerical variables are passed through unchanged because threshold-based trees do not require feature standardization. All models use the same 5 stratified folds (`shuffle=True`, `random_state=42`) on the training split.

In [6]:
model_preprocessor = ColumnTransformer([
    (
        "numeric",
        "passthrough",
        numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore"
        ),
        categorical_features
    )
])

In [7]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## 5. Untuned Decision Tree benchmark

Evaluate an unrestricted Decision Tree before regularization. The difference between its training ROC-AUC and cross-validation ROC-AUC indicates whether the model overfits.

In [8]:
decision_tree_pipeline = Pipeline([
    (
        "preprocessor",
        model_preprocessor
    ),
    (
        "model",
        DecisionTreeClassifier(
            random_state=42
        )
    )
])

In [9]:
baseline_dt_results = cross_validate(
    decision_tree_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1,
    return_train_score=True
)

In [10]:
baseline_dt_table = pd.DataFrame({
    "Fold": range(1, 6),
    "Train ROC-AUC": baseline_dt_results["train_roc_auc"],
    "CV ROC-AUC": baseline_dt_results["test_roc_auc"],
    "PR-AUC": baseline_dt_results["test_pr_auc"]
})

baseline_dt_table

,Fold,Train ROC-AUC,CV ROC-AUC,PR-AUC
0,1,0.999755,0.651128,0.365741
1,2,0.999711,0.652503,0.366568
2,3,0.999711,0.656840,0.370344
3,4,0.999724,0.649893,0.363423
4,5,0.999720,0.660137,0.375121


In [11]:
baseline_dt_roc = baseline_dt_results["test_roc_auc"].mean()
baseline_dt_std = baseline_dt_results["test_roc_auc"].std()

baseline_dt_pr = baseline_dt_results["test_pr_auc"].mean()

baseline_dt_train_roc = (
    baseline_dt_results["train_roc_auc"].mean()
)

print(
    "Mean Train ROC-AUC:",
    round(baseline_dt_train_roc, 4)
)

print(
    "Mean CV ROC-AUC:",
    round(baseline_dt_roc, 4)
)

print(
    "CV ROC-AUC std:",
    round(baseline_dt_std, 4)
)

print(
    "Mean CV PR-AUC:",
    round(baseline_dt_pr, 4)
)

Mean Train ROC-AUC: 0.9997
Mean CV ROC-AUC: 0.6541
CV ROC-AUC std: 0.0038
Mean CV PR-AUC: 0.3682


## 6. Tune tree complexity on the training split

`max_depth` limits the longest path through the tree; `min_samples_leaf` prevents fitting leaves to very small groups. The grid-search objective is **ROC-AUC only**. The separate validation and test splits are not passed to `GridSearchCV`.

In [12]:
param_grid = {
    "model__max_depth": [
        3,
        5,
        7,
        10,
        15,
        None
    ],
    "model__min_samples_leaf": [
        1,
        5,
        10,
        25,
        50
    ]
}

In [13]:
grid_search_dt = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search_dt.fit(
    X_train,
    y_train
)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         'passthrough',
                                                                         ['lead_time',
                                                                          'arrival_date_year',
                                                                          'arrival_date_week_number',
                                                                          'arrival_date_day_of_month',
                                                                          'stays_in_weekend_nights',
                                                                          'stays_in_week_nights',
                                                                          'adults',
                                                                          'children',
                                                                          'babies',
                                                                          'is_repeated...
                                                                          'arrival_date_month',
                                                                          'meal',
                                                                          'market_segment',
                                                                          'distribution_channel',
                                                                          'reserved_room_type',
                                                                          'deposit_type',
                                                                          'agent',
                                                                          'company',
                                                                          'season',
                                                                          'arrival_day_of_week'])])),
                                       ('model',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [3, 5, 7, 10, 15, None],
                         'model__min_samples_leaf': [1, 5, 10, 25, 50]},
             return_train_score=True, scoring='roc_auc')

In [14]:
print("Best parameters:")
print(grid_search_dt.best_params_)

print(
    "\nBest mean CV ROC-AUC:",
    round(grid_search_dt.best_score_, 4)
)

Best parameters:
{'model__max_depth': 10, 'model__min_samples_leaf': 10}

Best mean CV ROC-AUC: 0.7712


In [15]:
dt_tuning_results = pd.DataFrame(
    grid_search_dt.cv_results_
)

dt_tuning_table = dt_tuning_results[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "rank_test_score"
    ]
].copy()

dt_tuning_table = dt_tuning_table.rename(
    columns={
        "param_model__max_depth": "Max Depth",
        "param_model__min_samples_leaf": "Min Samples Leaf",
        "mean_test_score": "Mean CV ROC-AUC",
        "std_test_score": "CV ROC-AUC Std",
        "mean_train_score": "Mean Train ROC-AUC",
        "rank_test_score": "Rank"
    }
)

dt_tuning_table = dt_tuning_table.sort_values(
    "Rank"
).reset_index(drop=True)

dt_tuning_table.head(15)

,Max Depth,Min Samples Leaf,Mean CV ROC-AUC,CV ROC-AUC Std,Mean Train ROC-AUC,Rank
0,10,10,0.771235,0.004911,0.794767,1
1,15,50,0.771012,0.002286,0.811982,2
2,10,5,0.769941,0.005174,0.796433,3
3,10,1,0.769785,0.005372,0.797812,4
4,10,25,0.769674,0.003792,0.791191,5
5,10,50,0.769438,0.003841,0.787334,6
6,15,25,0.767829,0.003804,0.824653,7
7,None,50,0.766715,0.003609,0.826280,8
8,15,10,0.760922,0.004580,0.841001,9
9,7,5,0.760581,0.003510,0.768628,10


## 7. Training CV results and overfitting assessment

The grid search selects the configuration with the highest mean training CV ROC-AUC. The table below compares untuned and tuned train-to-CV gaps. A reduction in the gap supports better regularization but does not by itself prove performance on unseen data; the separate validation check appears next.

In [16]:
best_tree_pipeline = grid_search_dt.best_estimator_

final_dt_cv = cross_validate(
    best_tree_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1,
    return_train_score=True
)

final_dt_table = pd.DataFrame({
    "Fold": range(1, 6),
    "Train ROC-AUC": final_dt_cv["train_roc_auc"],
    "CV ROC-AUC": final_dt_cv["test_roc_auc"],
    "CV PR-AUC": final_dt_cv["test_pr_auc"]
})

final_dt_table

,Fold,Train ROC-AUC,CV ROC-AUC,CV PR-AUC
0,1,0.797336,0.765338,0.539362
1,2,0.795196,0.769322,0.542001
2,3,0.794483,0.770941,0.536796
3,4,0.793717,0.770326,0.542965
4,5,0.793103,0.780248,0.553329


In [17]:
final_dt_roc = final_dt_cv["test_roc_auc"].mean()
final_dt_roc_std = final_dt_cv["test_roc_auc"].std()

final_dt_pr = final_dt_cv["test_pr_auc"].mean()
final_dt_pr_std = final_dt_cv["test_pr_auc"].std()

final_dt_train_roc = final_dt_cv["train_roc_auc"].mean()

print("Final Train ROC-AUC:", round(final_dt_train_roc, 4))
print("Final CV ROC-AUC:", round(final_dt_roc, 4))
print("Final CV ROC-AUC std:", round(final_dt_roc_std, 4))
print("Final CV PR-AUC:", round(final_dt_pr, 4))
print("Final CV PR-AUC std:", round(final_dt_pr_std, 4))

Final Train ROC-AUC: 0.7948
Final CV ROC-AUC: 0.7712
Final CV ROC-AUC std: 0.0049
Final CV PR-AUC: 0.5429
Final CV PR-AUC std: 0.0056


In [18]:
tree_comparison = pd.DataFrame({
    "Model": [
        "Untuned Decision Tree",
        "Tuned Decision Tree"
    ],
    "Train ROC-AUC": [
        baseline_dt_train_roc,
        final_dt_train_roc
    ],
    "CV ROC-AUC": [
        baseline_dt_roc,
        final_dt_roc
    ],
    "CV PR-AUC": [
        baseline_dt_pr,
        final_dt_pr
    ]
})

tree_comparison["Train-CV Gap"] = (
    tree_comparison["Train ROC-AUC"]
    - tree_comparison["CV ROC-AUC"]
)

tree_comparison

,Model,Train ROC-AUC,CV ROC-AUC,CV PR-AUC,Train-CV Gap
0,Untuned Decision Tree,0.999724,0.654100,0.36824,0.345624
1,Tuned Decision Tree,0.794767,0.771235,0.54289,0.023532


In [19]:
# With refit=True, this estimator is already fitted on the complete train split.
assert grid_search_dt.refit is True
final_tree_model = grid_search_dt.best_estimator_
fitted_tree = final_tree_model.named_steps["model"]
assert hasattr(fitted_tree, "tree_")

print("Rows available to final refit:", len(X_train))
print("Rows represented by the final fitted tree:", int(fitted_tree.tree_.n_node_samples[0]))
assert int(fitted_tree.tree_.n_node_samples[0]) == len(X_train)
print("Best hyperparameters:", grid_search_dt.best_params_)
print("Best training CV ROC-AUC:", round(float(grid_search_dt.best_score_), 6))

Rows available to final refit: 58192
Rows represented by the final fitted tree: 58192
Best hyperparameters: {'model__max_depth': 10, 'model__min_samples_leaf': 10}
Best training CV ROC-AUC: 0.771235


## 8. Independent validation-set evaluation

The best model, already refitted on **all training observations**, is assessed on `hotel_val_prepared.csv` using the team's `src.evaluate.evaluate()` function. The threshold **0.5 is fixed and not tuned here**. ROC-AUC and PR-AUC use continuous scores; precision, recall, F1 and confusion matrix describe the fixed 0.5 decision threshold. Do not use these validation results for another parameter search.

In [20]:
DECISION_THRESHOLD = 0.5

tree_val_results = evaluate(
    final_tree_model,
    X_val,
    y_val,
    threshold=DECISION_THRESHOLD
)

# Older versions of evaluate() do not report accuracy explicitly.
val_cm = np.asarray(tree_val_results["confusion_matrix"])
val_accuracy = float(np.trace(val_cm) / val_cm.sum())
validation_metric_values = {
    "Accuracy": float(tree_val_results.get("accuracy", val_accuracy)),
    "ROC-AUC": float(tree_val_results["roc_auc"]),
    "PR-AUC": float(tree_val_results["pr_auc"]),
    "Precision @ 0.5": float(tree_val_results["precision"]),
    "Recall @ 0.5": float(tree_val_results["recall"]),
    "F1 @ 0.5": float(tree_val_results["f1"]),
}
validation_metrics = pd.DataFrame({
    "Metric": list(validation_metric_values),
    "Validation score": list(validation_metric_values.values())
})
display(validation_metrics)

confusion_table = pd.DataFrame(
    val_cm,
    index=["Actual 0 (not canceled)", "Actual 1 (canceled)"],
    columns=["Predicted 0", "Predicted 1"]
)
print("Validation confusion matrix (threshold = 0.5):")
display(confusion_table)

,Metric,Validation score
0,Accuracy,0.726621
1,ROC-AUC,0.754151
2,PR-AUC,0.520233
3,Precision @ 0.5,0.597729
4,Recall @ 0.5,0.237742
5,F1 @ 0.5,0.340180


Validation confusion matrix (threshold = 0.5):


,Predicted 0,Predicted 1
Actual 0 (not canceled),6862,496
Actual 1 (canceled),2363,737


## 9. Export readable decision rules

The fitted tree uses one-hot encoded feature names. Export a shortened rule preview for readability and a complete text file for the project handoff. The preview depth does **not** change the fitted tree's actual depth.

In [21]:
fitted_preprocessor = final_tree_model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print("Fitted tree depth:", fitted_tree.get_depth())
print("Leaf nodes:", fitted_tree.get_n_leaves())

print("Rule preview (limited for readability):")
print(export_text(fitted_tree, feature_names=list(feature_names), max_depth=5))

Number of transformed features: 85
Fitted tree depth: 10
Leaf nodes: 244
Rule preview (limited for readability):
|--- numeric__lead_time <= 11.50
|   |--- categorical__agent_9.0 <= 0.50
|   |   |--- numeric__previous_cancellations <= 0.50
|   |   |   |--- numeric__previous_bookings_not_canceled <= 0.50
|   |   |   |   |--- categorical__deposit_type_Non Refund <= 0.50
|   |   |   |   |   |--- numeric__lead_time <= 1.50
|   |   |   |   |   |   |--- truncated branch of depth 5
|   |   |   |   |   |--- numeric__lead_time >  1.50
|   |   |   |   |   |   |--- truncated branch of depth 5
|   |   |   |   |--- categorical__deposit_type_Non Refund >  0.50
|   |   |   |   |   |--- numeric__arrival_date_day_of_month <= 16.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- numeric__arrival_date_day_of_month >  16.50
|   |   |   |   |   |   |--- class: 1
|   |   |   |--- numeric__previous_bookings_not_canceled >  0.50
|   |   |   |   |--- categorical__hotel_Resort Hotel <= 0.50
|   | 

In [22]:
full_tree_rules = export_text(
    fitted_tree,
    feature_names=list(feature_names),
    max_depth=fitted_tree.get_depth()
)
RULES_PATH = MODEL_DIR / "decision_tree_rules.txt"
RULES_PATH.write_text(full_tree_rules, encoding="utf-8")
print("Complete readable rules saved to:", RULES_PATH)
print("Size (characters):", len(full_tree_rules))

Complete readable rules saved to: /content/hotel-cancellation/models/decision_tree_rules.txt
Size (characters): 45292


## 10. Why Decision Tree suits this problem

A Decision Tree can capture nonlinear patterns and interactions between booking characteristics, which may influence cancellation risk. It also provides explicit split rules that are useful for explaining business risk segments. Its complexity must be controlled because unrestricted trees can overfit, so tuning depth and minimum leaf size is essential.

## 11. Results for the team's Results sheet

Keep cross-validation metrics separate from the independent validation metrics. Model selection and final test evaluation follow the project's later-phase comparison protocol.

In [23]:
decision_tree_results = pd.DataFrame({
    "Model": ["Decision Tree"],
    "Configuration": [
        f"max_depth={grid_search_dt.best_params_['model__max_depth']}, "
        f"min_samples_leaf={grid_search_dt.best_params_['model__min_samples_leaf']}"
    ],
    "Train rows used": [len(X_train)],
    "CV ROC-AUC": [float(final_dt_roc)],
    "CV ROC-AUC Std": [float(final_dt_roc_std)],
    "CV PR-AUC": [float(final_dt_pr)],
    "CV PR-AUC Std": [float(final_dt_pr_std)],
    "Validation ROC-AUC": [float(tree_val_results["roc_auc"])],
    "Validation PR-AUC": [float(tree_val_results["pr_auc"])],
    "Validation Accuracy": [validation_metric_values["Accuracy"]],
    "Validation Precision @ 0.5": [float(tree_val_results["precision"])],
    "Validation Recall @ 0.5": [float(tree_val_results["recall"])],
    "Validation F1 @ 0.5": [float(tree_val_results["f1"])],
})
display(decision_tree_results)

,Model,Configuration,Train rows used,CV ROC-AUC,CV ROC-AUC Std,CV PR-AUC,CV PR-AUC Std,Validation ROC-AUC,Validation PR-AUC,Validation Accuracy,Validation Precision @ 0.5,Validation Recall @ 0.5,Validation F1 @ 0.5
0,Decision Tree,"max_depth=10, min_samples_leaf=10",58192,0.771235,0.004911,0.54289,0.005646,0.754151,0.520233,0.726621,0.597729,0.237742,0.34018


## 12. Save and verify the fitted model pipeline

The `.joblib` file includes the fitted one-hot encoder and Decision Tree. It expects the **27 shared-prepared predictor columns**, not raw bookings. The saved model was refitted on the existing full training split only; validation rows were used solely for evaluation.

In [24]:
MODEL_PATH = MODEL_DIR / "decision_tree_classifier.joblib"
joblib.dump(final_tree_model, MODEL_PATH)
loaded_tree = joblib.load(MODEL_PATH)

assert isinstance(loaded_tree, Pipeline)
assert set(loaded_tree.named_steps) == {"preprocessor", "model"}
np.testing.assert_allclose(
    loaded_tree.predict_proba(X_val.iloc[:10]),
    final_tree_model.predict_proba(X_val.iloc[:10]),
    rtol=0, atol=1e-12
)

print("Model saved and reloaded:", MODEL_PATH)
print("Rules file:", RULES_PATH)
print("Best tree hyperparameters:", grid_search_dt.best_params_)
print("Test split was not loaded.")

Model saved and reloaded: /content/hotel-cancellation/models/decision_tree_classifier.joblib
Rules file: /content/hotel-cancellation/models/decision_tree_rules.txt
Best tree hyperparameters: {'model__max_depth': 10, 'model__min_samples_leaf': 10}
Test split was not loaded.


## 13. Conclusion

The Decision Tree used the shared-prepared training split for baseline assessment, 5-fold CV and ROC-AUC-based hyperparameter tuning. `GridSearchCV(refit=True)` fitted the selected model on **all existing training rows**. It was then evaluated once on the separate validation split with the team's `evaluate()` function. No validation labels were used for selecting hyperparameters, and the test split remained untouched.

The output below reports the **actual scores from this run**. The joblib pipeline and complete human-readable rules are exported above. Threshold selection and final test-set comparison are left for later phases.

In [25]:
display(Markdown(
    f"**Training rows:** {len(X_train):,}. "
    f"**Best hyperparameters:** `{grid_search_dt.best_params_}`.  \n"
    f"**Train CV ROC-AUC:** {final_dt_roc:.4f} ± {final_dt_roc_std:.4f}. "
    f"**Train CV PR-AUC:** {final_dt_pr:.4f} ± {final_dt_pr_std:.4f}.  \n"
    f"**Validation ROC-AUC:** {tree_val_results['roc_auc']:.4f}. "
    f"**Validation PR-AUC:** {tree_val_results['pr_auc']:.4f}.  \n"
    f"**Validation precision / recall / F1 @ 0.5:** "
    f"{tree_val_results['precision']:.4f} / "
    f"{tree_val_results['recall']:.4f} / {tree_val_results['f1']:.4f}.  \n"
    "The difference between CV and validation ranking scores should be reported without retuning. "
    "Test data remains untouched."
))

**Training rows:** 58,192. **Best hyperparameters:** `{'model__max_depth': 10, 'model__min_samples_leaf': 10}`.  
**Train CV ROC-AUC:** 0.7712 ± 0.0049. **Train CV PR-AUC:** 0.5429 ± 0.0056.  
**Validation ROC-AUC:** 0.7542. **Validation PR-AUC:** 0.5202.  
**Validation precision / recall / F1 @ 0.5:** 0.5977 / 0.2377 / 0.3402.  
The difference between CV and validation ranking scores should be reported without retuning. Test data remains untouched.